In [1]:
import os

In [2]:
%pwd

'e:\\PROJECTS\\ML\\Text-Summarizer\\research'

In [3]:
%cd ..

e:\PROJECTS\ML\Text-Summarizer


c:\Users\reza\miniconda3\envs\textS\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
%pwd

'e:\\PROJECTS\\ML\\Text-Summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path
    

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self)->DataIngestionConfig:
        config = self.config.data_ingestion
        print(config)

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config
    

In [8]:
import os
import urllib.request as request
import zipfile
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size


In [11]:
class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    def download_file(self):
        """
        Downloads a file from the specified URL to the local directory if it does not already exist.

        Args:
            self: The object instance.
        
        Returns:
            None
        """
        if not os.path.exists(self.config.local_data_file):
            # Download the file from the source URL to the local directory
            filename, headers = request.urlretrieve(
                url=self.config.source_url,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with the following info: \n{headers}")
        else:
            # Log a message if the file already exists
            logger.info(f"File already exists with a size of: {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        """
        Extracts the contents of a zip file to the specified directory.

        Args:
            self: The object instance.
        
        Returns:
            None
        """
        # Create the directory if it does not exist
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        
        # Extract the zip file contents to the specified directory
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)



In [14]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2024-03-26 16:21:36,999: INFO: common : Yaml file: config\config.yaml loaded successfully]
[2024-03-26 16:21:37,003: INFO: common : Yaml file: params.yaml loaded successfully]
[2024-03-26 16:21:37,006: INFO: common : Created directory at: artifacts]
{'root_dir': 'artifacts/data_ingestion/', 'source_url': 'https://github.com/rezjsh/data/raw/main/samsum-data.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion/'}
[2024-03-26 16:21:37,010: INFO: common : Created directory at: artifacts/data_ingestion/]
[2024-03-26 16:21:37,011: INFO: 549889293 : File already exists with a size of: ~ 4036 KB]
